# Mood × RPE recovery checks

Specify one provisional model and test it on synthetic data.

First check recovery with clean ratings: no RPE effect, an additive RPE
effect, and a mood-dependent RPE effect. Then add slider mechanisms and
see what changes.

No real-data mood × RPE estimates here.

## 1. Defining the provisional model

### What I'm trying to estimate

Does the association between subsequent RPE exposure and the next happiness rating depend on prior mood?

This first iteration specifically tests whether the proposed estimator can recover a known state-dependent RPE effect, using synthetic data.

### Scope

- For this provisional model, I will use interval sums rather than a decaying history. Equal weights within an interval; earlier history is represented through prior mood and the play intercept, rather than explicit earlier reward terms.
- The fitted model uses observed ratings, not actual underlying moot. For the clean simulation these will be identical; later add a measurement layer.
- One signed RPE slope at each mood level, rather than separate slopes for positive and negative RPEs.

### Two parts of prior mood

Separate being above/below the play's usual rating from differences in average happiness between plays. Include both interactions with RPE, so the within-play term isn't forced to describe both things at once.

A play has 12 ratings: the pre-task rating ($H_{p,1}$) and 11 post-outcome ratings ($H_{p,2}$ through $H_{p,12}$).

Calculate the play mean from the 11 ratings that serve as prior mood: the pre-task rating ($H_{p,1}$) through the second-to-last rating ($H_{p,11}$). The final rating ($H_{p,12}$) is only an outcome, so leave it out of this mean. This makes $M^W$ average zero across the play's 11 intervals. “Prior” describes the rating's role in an interval, not a separate set of measurements.

$$\bar H_p=\frac{1}{11}\sum_{k=1}^{11}H_{p,k},\qquad M^W_{p,k}=H_{p,k}-\bar H_p,\qquad M^B_p=\bar H_p-50.$$

$M^W$ is the within-play deviation; $M^B$ is the play's average relative to 50. This isn't z-scoring: no division by the SD. The mean includes ratings that are also outcomes in the model, so the centred prior-mood predictor and outcomes share information by construction. Test what this does to $\rho$ and $\theta_W$ with only 11 intervals. Averaging to zero doesn't by itself remove estimation bias.

### The provisional model

$$H_{p,k+1}=\alpha_p+\rho M^W_{p,k}+\beta_{\mathrm{CR}}CR_{p,k}+\beta_{\mathrm{EV}}EV_{p,k}+\beta_{\mathrm{RPE}}RPE_{p,k}+\theta_W M^W_{p,k}RPE_{p,k}+\theta_B M^B_pRPE_{p,k}+\tau_k+\varepsilon_{p,k}.$$

### Reading the terms

- $H_{p,k}$: **observed happiness rating**, in rating points. The real scale is 0–100. In the clean simulation set $H_{p,k}=L_{p,k}$, where $L$ is latent mood; later add slider start $S$ to the measurement layer. How to handle boundaries is a separate simulation choice.
- $p$: **play** (technically participant-and-play pair).
- $k$: **interval index**, $k=1,\dots,11$. Interval $k$ runs from rating $H_{p,k}$ to rating $H_{p,k+1}$, so its prior mood comes from $H_{p,k}$ and its outcome is $H_{p,k+1}$. $H_{p,1}$ is the pre-task rating and $H_{p,12}$ the final rating.
- $M^W_{p,k}$: **prior mood relative to this play's average**. Negative means below that average, not necessarily below 50.
- $M^B_p$: **average prior rating relative to 50**. Constant within a play. Its additive effect is absorbed by the play intercept, but its interaction with RPE isn't.
- $CR_{p,k}$, $EV_{p,k}$, $RPE_{p,k}$: certain rewards, chosen-gamble expected values and RPEs, all in game points and summed over the interval. Their order within the interval is not represented.
- $\alpha_p$: **play fixed effect**; a separate intercept allowing plays to differ in overall rating level. Could reflect (in part) how happy this person tends to be in this play, where on the scale they tend to rate, etc. It does not absorb differences in scale sensitivity or reward slopes. Distinct from the pre-task rating and from the raw play average.
- $\rho$: **carry-over**; the prior-rating association when interval RPE is zero. With the interaction, the within-play association is $\rho+\theta_W RPE$, treating the play mean as fixed. It may reflect more than mood persistence.
- $\beta_{\mathrm{CR}},\beta_{\mathrm{EV}},\beta_{\mathrm{RPE}}$: **reward weights**. Comparable in role to Rutledge's $w_1$–$w_3$, not identical. CR and EV slopes don't vary with mood in this first model.
- $\theta_W$: **the interaction I'm interested in**; change in RPE slope per rating point above/below the play's average.
- $\theta_B$: allows plays averaging higher happiness to have different RPE slopes. This is between-play, not necessarily between-person, and doesn't absorb every possible slope difference.
- $\tau_k$: **interval-position effect**; a shift for each position (one as reference), capturing a time pattern common to everyone, not each person's own drift or changes in RPE sensitivity over time.
- $\varepsilon_{p,k}$: residual; everything the model doesn't explain.

The RPE slope is now:

$$\beta_{\mathrm{RPE}}+\theta_W M^W_{p,k}+\theta_B M^B_p.$$

So $\beta_{\mathrm{RPE}}$ refers to a play averaging 50, at its own average mood. Uses the same signed slope for positive and negative RPEs. While that slope stays positive, positive $\theta_W$ means equally sized positive and negative surprises have larger contributions at higher within-play mood. It doesn't estimate separate positive/negative sensitivities.

The old midpoint-centred model tied the two interactions to the same coefficient. Here they can differ. No need for the full derivation in these notes.

### Worked example (invented values)

Take a play whose prior ratings average 50, so $M^B_p=0$. With $\beta_{\mathrm{RPE}}=0.05$ and $\theta_W=0.001$:

| Prior mood relative to play average | $M^W$ | RPE slope | Contribution of RPE = +20 |
|---|---|---|---|
| 20 points below | −20 | 0.03 | +0.6 |
| At the average | 0 | 0.05 | +1.0 |
| 20 points above | +20 | 0.07 | +1.4 |

An RPE of −20 gives the same amounts but negative, holding the other terms fixed. For a play averaging 70, a rating of 60 is instead $M^W=-10$ and $M^B=20$; its slope also includes $20\theta_B$.

Because gamble RPE is outcome minus EV, the interaction ties mood's outcome and expectation interactions to opposite coefficients. This is a model of the RPE–rating association, not evidence about a neural mechanism.

### Choices to test

- Keep a fixed happiness scale for now. Standardising within a play uses later ratings as well as earlier ones and changes the scale of the interaction. Leave that out of the first recovery check.
- Keeping a separate intercept for each play.
- Using baseline (pre-task rating) as prior mood for the first interval. Later ratings are outcomes once and predictors in the next row.
- The separate CR, EV and RPE terms follow Rutledge's general approach, but this isn't model 1: it adds prior mood and interactions, and uses interval sums instead of exponential history.
- Play fixed effects plus a previous-rating predictor can be biased with only 11 intervals. Check recovery on clean synthetic data before adding slider effects. Clustered SEs don't fix coefficient bias.
- Build the interaction products before subtracting play means for fixed-effects fitting. Multiplying separately demeaned components isn't generally the same thing.
- Also check scale boundaries. A rating near 0 or 100 has less room to move in one direction, which could look like mood-dependent sensitivity.

### Generating mood versus fitting ratings

For the clean check, generate latent mood forward from a known typical level for each play, then set observed ratings equal to latent mood. Don't generate from the realised whole-play mean: that only exists after the sequence is finished.

After generating the sequence, calculate the mean of the 11 observed ratings used as prior mood ($H_{p,1}$ through $H_{p,11}$) and use it in the fitted model. Also try the known simulated typical level as a benchmark, to separate estimating the mean from other fitting issues.

Later the slider layer changes the observed rating, not automatically the latent mood used to generate the next state. Keep those two sequences separate.

### Order of checks

1. Clean, noisy simulations with null and nonzero within-play moderation. Compare the decomposition with the midpoint-centred model over repeated simulated datasets.
2. Between-play RPE slope differences linked to typical mood.
3. Fading RPE sensitivity across positions; position intercepts alone don't handle this.
4. Decaying reward history, scale bounds and slider mechanisms, one departure at a time as time allows.

Random RPE slopes by participant can wait. No real-data mood × RPE estimate before settling the specification.